In [7]:
from os import path
import polars as pl
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import pandas as pd

tabular_data_path = path.join("..", "..", "data", "tabular")
processed_csv = path.join(tabular_data_path, "processed.csv")
target_col = "GUY SCORE"

df = pl.read_csv(processed_csv)

preop_features = [
    "LOKALİZASYON",
    "TOPLAM TAŞ YÜKÜ (CM2)",
    "YAŞ",
]

preop_features = [c for c in preop_features if c in df.columns]

X = df.select(preop_features).to_numpy()
y = df[target_col].cast(pl.Int64, strict=False).to_numpy() - 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=492, stratify=y
)

dt_model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=492,
    criterion="gini",
)

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print("=" * 60)
print("GUY'S STONE SCORE - CLINICAL DECISION TREE")
print("=" * 60)

print(f"\nTest Set: {len(y_test)} patients")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(f"Weighted F1: {f1_score(y_test, y_pred, average='weighted'):.2f}")

cv_scores = cross_val_score(dt_model, X_train, y_train, cv=5, scoring="f1_weighted")
print(f"5-Fold CV F1: {cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")

print("\n" + "=" * 60)
print("DECISION RULES")
print("=" * 60)

tree_rules = export_text(
    dt_model,
    feature_names=preop_features,
    class_names=["Grade 1", "Grade 2", "Grade 3", "Grade 4"],
    show_weights=True,
)
print(tree_rules)

print("=" * 60)
print("CLINICAL LOOKUP TABLE")
print("=" * 60)

print("""
LOKALİZASYON Codes:
  1  = Üst kaliks          → Grade 2
  2  = Orta kaliks         → Grade 1
  3  = Alt kaliks          → Grade 1
  4  = Pelvis              → Grade 1
  5  = Parsiyel staghorn   → Grade 3
  11 = Staghorn (tam)      → Grade 4
  20 = Multipl lokalizasyon → Grade 2 (or 3 if high burden)

Simple Formula:
─────────────────────────────────────
IF LOKALİZASYON = 11        → Grade 4
IF LOKALİZASYON = 5         → Grade 3
IF LOKALİZASYON = 1         → Grade 2
IF LOKALİZASYON = 2,3,4     → Grade 1
IF LOKALİZASYON = 20        → Check stone burden + age
─────────────────────────────────────
""")
plt.style.use('default')
fig, ax = plt.subplots(figsize=(22, 14), facecolor='white')
ax.set_facecolor('white')

plot_tree(
    dt_model,
    feature_names=preop_features,
    class_names=["Grade 1", "Grade 2", "Grade 3", "Grade 4"],
    filled=True,
    rounded=True,
    fontsize=11,
    ax=ax,
    impurity=False,
    proportion=True,
    node_ids=True,
)

ax.set_title("Guy's Stone Score - Decision Tree", fontsize=16, pad=20, fontweight='bold')

plt.savefig("gss_decision_tree.png", dpi=300, bbox_inches="tight", facecolor='white')
plt.close()

print("\nDecision tree saved: gss_decision_tree.png")

feature_importance = pd.DataFrame(
    {"feature": preop_features, "importance": dt_model.feature_importances_}
).sort_values("importance", ascending=False)

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)
print(feature_importance.to_string(index=False))

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Grade 1", "Grade 2", "Grade 3", "Grade 4"], digits=3))

GUY'S STONE SCORE - CLINICAL DECISION TREE

Test Set: 54 patients
Accuracy: 0.80
Weighted F1: 0.79
5-Fold CV F1: 0.77 (+/- 0.07)

DECISION RULES
|--- LOKALİZASYON <= 15.50
|   |--- LOKALİZASYON <= 8.00
|   |   |--- LOKALİZASYON <= 4.50
|   |   |   |--- weights: [50.00, 22.00, 3.00, 0.00] class: Grade 1
|   |   |--- LOKALİZASYON >  4.50
|   |   |   |--- weights: [0.00, 0.00, 14.00, 0.00] class: Grade 3
|   |--- LOKALİZASYON >  8.00
|   |   |--- TOPLAM TAŞ YÜKÜ (CM2) <= 18.00
|   |   |   |--- weights: [0.00, 1.00, 0.00, 29.00] class: Grade 4
|   |   |--- TOPLAM TAŞ YÜKÜ (CM2) >  18.00
|   |   |   |--- weights: [0.00, 0.00, 1.00, 4.00] class: Grade 4
|--- LOKALİZASYON >  15.50
|   |--- YAŞ <= 16.50
|   |   |--- TOPLAM TAŞ YÜKÜ (CM2) <= 1.00
|   |   |   |--- weights: [2.00, 17.00, 5.00, 3.00] class: Grade 2
|   |   |--- TOPLAM TAŞ YÜKÜ (CM2) >  1.00
|   |   |   |--- weights: [0.00, 51.00, 6.00, 1.00] class: Grade 2
|   |--- YAŞ >  16.50
|   |   |--- weights: [0.00, 2.00, 4.00, 0.00] class: